# Second round checkpoints: 4000 samples, a real three way split

The first benchmark explained 100 reviews per dataset from checkpoints with a
plain 80/20 split. This round doubles the data and adopts the standard
train, validation, test protocol at 70/15/15: 2800 training documents, 600
validation, 600 test per dataset. The head never sees validation or test, its
early stopping uses an internal slice of the training set, so validation
stays clean for any future model selection and the test split exists solely
for explanation evaluation. Everything else about the pipeline, encoder,
head, cleaning, filters and seeds, is unchanged, so differences against the
first round come from the data volume and the split, not from modeling
drift.

The new checkpoints live beside the old ones, `imdb_4000`, `yelp_4000` and
`agnews_4000`, and the old ones stay untouched so every earlier notebook
keeps reproducing.

## Configuration

In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIRECTORY = PROJECT_ROOT / "data"
ARTIFACTS_DIRECTORY = PROJECT_ROOT / "artifacts"

SAMPLE_SIZE = 4000
TEST_FRACTION = 0.15
VALIDATION_FRACTION = 0.15
POOL_SIZE = 20000
RANDOM_SEED = 42

print(f"Sample size: {SAMPLE_SIZE}, split 70/15/15 -> "
      f"{int(SAMPLE_SIZE * 0.7)}/{int(SAMPLE_SIZE * 0.15)}/{int(SAMPLE_SIZE * 0.15)}")

Sample size: 4000, split 70/15/15 -> 2800/600/600


## Load the three corpora into pools

IMDB comes from the local Kaggle export notebook 01 downloaded, Yelp and
AG News from the fast.ai archives notebook 05 extracted. Each corpus is
cleaned and reduced to a seeded stratified pool of 20,000 rows, from which
`build_checkpoint` draws its stratified 4000.

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

from textfaith.training import clean_text

imdb_raw = pd.read_csv(DATA_DIRECTORY / "IMDB Dataset.csv")
imdb_frame = pd.DataFrame(
    {
        "text": imdb_raw["review"].map(clean_text),
        "label": (imdb_raw["sentiment"] == "positive").astype(int),
    }
)

agnews_raw = pd.read_csv(
    DATA_DIRECTORY / "ag_news_csv" / "train.csv",
    header=None,
    names=["label", "title", "description"],
)
agnews_binary = agnews_raw[agnews_raw["label"].isin([3, 4])].copy()
agnews_frame = pd.DataFrame(
    {
        "text": (agnews_binary["title"] + " " + agnews_binary["description"]).map(clean_text),
        "label": (agnews_binary["label"] == 4).astype(int),
    }
)

yelp_raw = pd.read_csv(
    DATA_DIRECTORY / "yelp_review_polarity_csv" / "train.csv",
    header=None,
    names=["label", "text"],
)
yelp_frame = pd.DataFrame(
    {
        "text": yelp_raw["text"].map(clean_text),
        "label": (yelp_raw["label"] == 2).astype(int),
    }
)

pools = {}
for name, frame in [("imdb", imdb_frame), ("yelp", yelp_frame), ("agnews", agnews_frame)]:
    pools[name], _ = train_test_split(
        frame,
        train_size=POOL_SIZE,
        random_state=RANDOM_SEED,
        stratify=frame["label"],
    )
    print(f"{name}: pool of {len(pools[name])} from {len(frame)}, "
          f"balance {pools[name]['label'].value_counts().to_dict()}")

/home/user/projects/shapley/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


imdb: pool of 20000 from 50000, balance {1: 10000, 0: 10000}
yelp: pool of 20000 from 560000, balance {1: 10000, 0: 10000}
agnews: pool of 20000 from 60000, balance {1: 10000, 0: 10000}


## Train the three checkpoints

Same `build_checkpoint`, now with `validation_fraction`: test is carved out
first, validation from the remainder, both stratified on the label.

In [3]:
from textfaith.training import build_checkpoint

LABEL_NAMES = {
    "imdb": {0: "negative", 1: "positive"},
    "yelp": {0: "negative", 1: "positive"},
    "agnews": {0: "business", 1: "sci/tech"},
}

metadata_by_dataset = {}
for name in ["imdb", "yelp", "agnews"]:
    metadata_by_dataset[name] = build_checkpoint(
        pools[name],
        ARTIFACTS_DIRECTORY / f"{name}_4000",
        dataset_name=name,
        label_names=LABEL_NAMES[name],
        sample_size=SAMPLE_SIZE,
        test_fraction=TEST_FRACTION,
        validation_fraction=VALIDATION_FRACTION,
        random_seed=RANDOM_SEED,
    )
    metadata = metadata_by_dataset[name]
    print(
        f"{name}: train {metadata['train_size']}, "
        f"validation {metadata['validation_size']}, test {metadata['test_size']}, "
        f"validation accuracy {metadata['validation_accuracy']:.4f}, "
        f"test accuracy {metadata['test_accuracy']:.4f}"
    )

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 18157.95it/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (641 > 512). Running this sequence through the model will result in indexing errors


imdb: train 2800, validation 600, test 600, validation accuracy 0.9067, test accuracy 0.9350


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

...

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (943 > 512). Running this sequence through the model will result in indexing errors


yelp: train 2800, validation 600, test 600, validation accuracy 0.9700, test accuracy 0.9567


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 19134.95it/s]

agnews: train 2800, validation 600, test 600, validation accuracy 0.9050, test accuracy 0.8917


## Round trip through the predictor

The benchmark consumes these checkpoints through the unmodified
`load_predictor`, so that path is what gets verified: each checkpoint's test
split re-scored through the loaded predictor must reproduce the recorded
accuracy exactly.

In [4]:
import json

import numpy as np

from textfaith.checkpoint import load_predictor

summary_rows = []
for name in ["imdb", "yelp", "agnews"]:
    directory = ARTIFACTS_DIRECTORY / f"{name}_4000"
    predict_probabilities, _ = load_predictor(directory)
    test_reviews = pd.read_parquet(directory / "test.parquet")
    predictions = predict_probabilities(test_reviews["text"].tolist()).argmax(axis=1)
    accuracy = float((predictions == test_reviews["label"].to_numpy()).mean())
    metadata = json.loads((directory / "metadata.json").read_text())
    assert np.isclose(accuracy, metadata["test_accuracy"], atol=1e-9), (
        f"{name}: loaded predictor accuracy {accuracy} does not reproduce "
        f"the recorded {metadata['test_accuracy']}"
    )
    summary_rows.append(
        {
            "dataset": name,
            "train": metadata["train_size"],
            "validation": metadata["validation_size"],
            "test": metadata["test_size"],
            "validation_accuracy": metadata["validation_accuracy"],
            "test_accuracy": metadata["test_accuracy"],
            "median_test_words": int(
                test_reviews["text"].str.split().str.len().median()
            ),
        }
    )

summary = pd.DataFrame(summary_rows).set_index("dataset")
print("Every checkpoint reproduces its recorded accuracy through load_predictor")
summary

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 22316.69it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 18711.56it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 21184.97it/s]

Every checkpoint reproduces its recorded accuracy through load_predictor


,train,validation,test,validation_accuracy,test_accuracy,median_test_words
dataset,,,,,,
imdb,2800,600,600,0.906667,0.935000,157
yelp,2800,600,600,0.970000,0.956667,103
agnews,2800,600,600,0.905000,0.891667,38
